# 2. Route workbench

A control panel for generating routes. Change the parameters in the next cell, run the notebook, and
watch the search converge on a target distance before seeing the results on a map.

Everything here calls the same code the app will. Nothing is reimplemented, so a route that looks
good in this notebook is a route the app will produce.

## Parameters

The only cell you need to edit.

In [ ]:
TARGET_KM      = 10          # how far you want to run
ATTEMPTS       = 24          # bearings to try. more attempts, more candidates, slower
TOLERANCE      = 0.10        # how close to TARGET_KM a route must land, as a fraction
ROUNDS         = 5           # how many times to correct the radius before giving up on a bearing
SEED           = 1           # fixes which bearings are tried, so a run is repeatable
STRATEGY_NAME  = 'avoid_roads'
SHOW_TOP       = 3           # routes to draw on the map

MAP_WIDTH_PX   = 1000        # Leaflet sizes itself on start, so the map needs explicit dimensions
MAP_HEIGHT_PX  = 650

In [ ]:
import time
from collections import defaultdict

import folium
import matplotlib.pyplot as plt
import osmnx

from pathrun import config, cost, gpx, loops, network, strategy

# The reference categorical palette. Only the first three slots are used, which is the documented
# limit for charts where every pair of series can appear together.
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#d8d8d4'

chosen = strategy.STRATEGIES[STRATEGY_NAME]
print(f'{chosen.name}: {chosen.description}')

## The network

Loaded from the cached copy, then costed under the chosen strategy. Costing removes ways you are not
allowed on or would never want, so the edge count drops.

In [ ]:
graph = network.load_network(config.HOME_LATITUDE, config.HOME_LONGITUDE, config.NETWORK_RADIUS_METRES)
before = graph.number_of_edges()
graph = cost.apply_costs(graph, chosen)
home = osmnx.distance.nearest_nodes(graph, X=config.HOME_LONGITUDE, Y=config.HOME_LATITUDE)
scale = cost.observed_minimum_penalty(graph)
print(f'{before:,} edges, {before - graph.number_of_edges():,} removed as unusable')
print(f'home node {home}, A* heuristic scale {scale:.4f}')

## Watching the search converge

Placing a waypoint by straight line distance badly underestimates how long the route will be, because
a strategy that avoids roads sends the route wandering. So the search measures what came back and
scales the radius by how wrong it was, then tries again.

Each line below is one attempt. Watch the error column fall.

In [ ]:
attempts = []

def watch(record):
    attempts.append(record)
    print(f"  bearing {record['bearing']:5.0f}  round {record['round']}  "
          f"radius {record['radius_metres']:6.0f} m  ->  {record['length_metres']/1000:6.2f} km  "
          f"error {100*record['error']:5.1f}%")

started = time.time()
candidates = loops.generate_loops(graph, home, TARGET_KM, attempts=ATTEMPTS,
                                  tolerance=TOLERANCE, seed=SEED, observer=watch)
print(f'\n{len(candidates)} usable loops from {ATTEMPTS} bearings, {len(attempts)} attempts, {time.time()-started:.0f}s')

### How quickly it converges

One line per bearing, showing the route length each correction produced. The dashed line is the
target. A bearing that leaves the shaded band never found an acceptable route and was dropped.

In [ ]:
by_bearing = defaultdict(list)
for record in attempts: by_bearing[record['bearing']].append(record)

figure, axis = plt.subplots(figsize=(9, 5))
axis.axhspan(TARGET_KM*(1-TOLERANCE), TARGET_KM*(1+TOLERANCE), color=SERIES[0], alpha=0.10, zorder=0)
axis.axhline(TARGET_KM, color=MUTED, linestyle='--', linewidth=2, zorder=2)
for records in by_bearing.values():
    lengths = [record['length_metres']/1000 for record in records]
    axis.plot(range(len(lengths)), lengths, color=SERIES[0], alpha=0.45, linewidth=2, marker='o', markersize=5, zorder=3)
axis.set_xlabel('correction round'); axis.set_ylabel('route length, km')
axis.set_title(f'Each bearing corrects towards {TARGET_KM} km', color=INK, loc='left')
axis.set_xticks(range(ROUNDS))
axis.spines[['top','right']].set_visible(False)
axis.spines[['left','bottom']].set_color(GRID)
axis.tick_params(colors=MUTED)
axis.grid(axis='y', color=GRID, linewidth=0.8)
axis.set_axisbelow(True)
plt.show()

### What the candidates look like

Each dot is a finished route. Further right is more of the run on a legally protected right of way,
which is the thing this strategy exists to maximise. Lower is less ground covered twice. So the
bottom right corner is where you want to be.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5.5))
for index, generator_name in enumerate(loops.GENERATORS):
    matching = [candidate for candidate in candidates if candidate['generator'] == generator_name]
    if not matching: continue
    axis.scatter([100*candidate['score']['right_of_way_share'] for candidate in matching],
                 [100*candidate['score']['repeated_share'] for candidate in matching],
                 s=110, color=SERIES[index], edgecolor='white', linewidth=2, label=generator_name, zorder=3)
axis.set_xlabel('on rights of way, %'); axis.set_ylabel('ground covered twice, %')
axis.set_title('Bottom right is best', color=INK, loc='left')
axis.legend(frameon=False, labelcolor=MUTED)
axis.spines[['top','right']].set_visible(False)
axis.spines[['left','bottom']].set_color(GRID)
axis.tick_params(colors=MUTED)
axis.grid(color=GRID, linewidth=0.8)
axis.set_axisbelow(True)
plt.show()

## The routes on a map

The top candidates over a real map. Use the layer control in the top right to show one at a time.

The map is also written to `data/routes/` as a standalone HTML file. Open that in a browser if the
inline version looks grey, which happens when a stored notebook output is displayed at a different
size from the one it was drawn at.

In [ ]:
top = candidates[:SHOW_TOP]
for rank, candidate in enumerate(top, start=1):
    score = candidate['score']
    print(f"{rank}. {score['total_km']:5.2f} km   rights of way {100*score['right_of_way_share']:3.0f}%   "
          f"repeated {100*score['repeated_share']:3.0f}%   {candidate['generator']}")

In [ ]:
# Leaflet measures its container when it starts, so the figure is given an explicit size rather than
# left to inherit one. Without it a headless run requests tiles for a container of unknown size and
# most of the map comes back grey.
figure = folium.Figure(width=MAP_WIDTH_PX, height=MAP_HEIGHT_PX)
route_map = folium.Map(tiles='OpenStreetMap', control_scale=True)
route_map.add_to(figure)

folium.Marker([config.HOME_LATITUDE, config.HOME_LONGITUDE], tooltip='home',
              icon=folium.Icon(color='black', icon='home', prefix='fa')).add_to(route_map)

all_points = []
for rank, (candidate, colour) in enumerate(zip(top, SERIES), start=1):
    score = candidate['score']
    points = gpx.route_points(graph, candidate['route'])
    all_points += points
    label = (f"{rank}. {score['total_km']:.2f} km, "
             f"{100*score['right_of_way_share']:.0f}% rights of way, "
             f"{100*score['repeated_share']:.0f}% repeated")
    layer = folium.FeatureGroup(name=label, show=True)
    folium.PolyLine(points, color=colour, weight=5, opacity=0.85, tooltip=label).add_to(layer)
    layer.add_to(route_map)

folium.LayerControl(collapsed=False).add_to(route_map)
# Frame the routes rather than trusting a fixed zoom, which cuts long loops off the edge.
route_map.fit_bounds([[min(latitude for latitude, _ in all_points), min(longitude for _, longitude in all_points)],
                      [max(latitude for latitude, _ in all_points), max(longitude for _, longitude in all_points)]])

map_path = config.CACHE_DIRECTORY/'routes'/f'map_{TARGET_KM:g}km.html'
map_path.parent.mkdir(parents=True, exist_ok=True)
figure.save(str(map_path))
print(f'saved to {map_path}')
print('If the inline map below is grey, open that file in a browser. A stored notebook output cannot')
print('resize a Leaflet map that was drawn in a headless kernel.')
figure

## Export the one you want

Change `PICK` to the rank you liked, run the cell, and the GPX lands in `data/routes/`.

Strava cannot create a route through its API, so the last step is manual and takes two clicks:
Dashboard, then My Routes, then Create New Route, then the upload button. If you use a Garmin, Garmin
Connect imports the same file as a course and syncs it straight to the watch.

In [ ]:
PICK = 1

chosen_route = top[PICK-1]
written = gpx.write_route(graph, chosen_route['route'],
                          config.CACHE_DIRECTORY/'routes'/f'loop_{TARGET_KM:g}km_{PICK}.gpx',
                          name=f"pathrun {chosen_route['score']['total_km']:.1f} km loop")
print(f'wrote {written}')
cost.print_score(chosen_route['score'])

## Trying a different strategy

`AVOID_ROADS` is the only strategy today. To try different weights, build one and pass it to
`cost.apply_costs`. Nothing else changes, which is the point of keeping the weights in a strategy
rather than in the routing code.

```python
from dataclasses import replace
gentler = replace(strategy.AVOID_ROADS, name='gentler', right_of_way_discount=0.9)
graph = cost.apply_costs(graph, gentler)
```

Note that `apply_costs` removes excluded edges, so reload the network from cache before costing it
under a strategy that excludes less than the last one did.